# 08 — ASR and Content Representation

## Objective

Prepare the **original/anonymized speech pairs** required for content-preservation evaluation.

The anonymization method is intentionally left open. Whatever route is chosen later must produce an anonymized audio file paired with its corresponding original audio.

$$
(X_o,X_a)\xrightarrow{\mathrm{ASR}}(W_o,W_a)
$$


**Principles:** same ASR system/configuration on both sides; preserve raw ASR transcripts; do not choose an anonymization route here; do not combine metrics into a final loss here.

In [1]:
import sys
from pathlib import Path

root = Path.cwd().parent.resolve()
print("Root:", root)
print("In sys.path:", str(root) in sys.path)
print("src package:", (root / "src" / "__init__.py").is_file())

import importlib.util
print("src spec:", importlib.util.find_spec("src"))

Root: /Users/juni/Documents/interview-anonymization
In sys.path: False
src package: True
src spec: ModuleSpec(name='src', loader=None, submodule_search_locations=_NamespacePath(['/opt/anaconda3/envs/interview-anon/lib/python3.11/site-packages/src']))


In [2]:
import sys
from pathlib import Path

import pandas as pd
from faster_whisper import WhisperModel

# Locate the project root independently of the working directory.
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in (cwd, *cwd.parents)
        if (path / "src" / "paths.py").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root.")

# Load the utility module directly from its file path.
import importlib.util

utils_spec = importlib.util.spec_from_file_location(
    "project_utils",
    PROJECT_ROOT / "src" / "utils.py",
)

if utils_spec is None or utils_spec.loader is None:
    raise ImportError("Could not load src/utils.py.")

utils = importlib.util.module_from_spec(utils_spec)
utils_spec.loader.exec_module(utils)

# Load centralized project paths using the utility function.
paths = utils.load_module(
    "project_paths",
    PROJECT_ROOT / "src" / "paths.py",
)

PROJECT_ROOT = paths.PROJECT_ROOT
METADATA_DIR = paths.METADATA_DIR
RESULTS_DIR = paths.RESULTS_DIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Metadata directory:", METADATA_DIR)
print("Results directory:", RESULTS_DIR)

Project root: /Users/juni/Documents/interview-anonymization
Metadata directory: /Users/juni/Documents/interview-anonymization/data/metadata
Results directory: /Users/juni/Documents/interview-anonymization/data/results


/opt/anaconda3/envs/interview-anon/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. ASR configuration

Faster-Whisper is used to obtain $W_o$ and $W_a$.

The **same model and decoding configuration** must be used for original and anonymized audio so that transcript differences are not caused by changing the ASR system.

In [3]:
ASR_MODEL_NAME = "base"
ASR_DEVICE = "cpu"
ASR_COMPUTE_TYPE = "int8"
ASR_LANGUAGE = "en"

asr_model = WhisperModel(
    ASR_MODEL_NAME,
    device=ASR_DEVICE,
    compute_type=ASR_COMPUTE_TYPE,
)

print("ASR model:", ASR_MODEL_NAME)
print("Device:", ASR_DEVICE)
print("Language:", ASR_LANGUAGE)


ASR model: base
Device: cpu
Language: en


## 2. Content encoder

The encoder converts each ASR transcript into a semantic representation.

Keep the model selected in the dedicated model-selection notebook and use the **same encoder for both sides**.

These embeddings support semantic similarity. They are not a replacement for NLI or TPI.

## 3. ASR helper

Keep the transcript as produced by ASR. Do not apply semantic corrections or aggressive normalization at this stage.

In [4]:
asr = utils.load_module(
    "project_asr",
    PROJECT_ROOT / "src" / "asr.py",
)

transcribe_audio = asr.transcribe_audio

## 4. ASR sanity check

VoxCeleb is used here only to verify that the ASR pipeline works. This is **not yet** the content-preservation experiment, because that experiment requires an original/anonymized pair.

In [5]:
metadata_path = METADATA_DIR / "voxceleb_utterances.parquet"

if not metadata_path.is_file():
    raise FileNotFoundError(
        f"VoxCeleb metadata not found: {metadata_path}"
    )

utterance_df = pd.read_parquet(metadata_path)

if utterance_df.empty:
    raise ValueError("VoxCeleb metadata contains no utterances.")

if "path" not in utterance_df.columns:
    raise ValueError(
        "VoxCeleb metadata must contain a 'path' column."
    )

# Resolve relative audio paths against the VoxCeleb WAV directory.
wav_files = [
    paths.VOXCELEB_WAV_DIR / Path(path)
    for path in utterance_df["path"]
]

print("Number of WAV files:", len(wav_files))

sample_path = wav_files[0]
print("Sample:", sample_path)
print("Exists:", sample_path.is_file())

Number of WAV files: 124843
Sample: /Users/juni/Documents/interview-anonymization/data/raw/voxceleb1/wav/id10001/1zcIwhmdeo4/00001.wav
Exists: True


In [6]:
print("Number of utterances:", len(utterance_df))
print("Number of speakers:", utterance_df["speaker_id"].nunique())

sample = utterance_df.iloc[0]
audio_path = paths.VOXCELEB_WAV_DIR / sample["path"]

print("Speaker:", sample["speaker_id"])
print("Video:", sample["video_id"])
print("Split:", sample["split"])
print("Language:", sample["language"])
print("Audio:", audio_path)
print("Exists:", audio_path.exists())


Number of utterances: 124843
Number of speakers: 1166
Speaker: id10001
Video: 1zcIwhmdeo4
Split: train
Language: english
Audio: /Users/juni/Documents/interview-anonymization/data/raw/voxceleb1/wav/id10001/1zcIwhmdeo4/00001.wav
Exists: True


In [7]:
sample_transcription = transcribe_audio(
    audio_path,
    model=asr_model,
    language=ASR_LANGUAGE,
)

print("ASR transcription:")
print(sample_transcription)


ASR transcription:
Looking guys up that lived in the Everglades and stuff and and that my character was described as a romantic sociopath who was good with a knife so I figured


## 5. Pair-level data interface

The real experiment needs a table like:

| pair_id | original_path | anonymized_path |
|---|---|---|
| ... | original audio | anonymized audio |

The anonymization route is **not specified here**. Once a route is selected, its outputs are registered as `anonymized_path` for the corresponding `pair_id`.

Additional metadata such as speaker, split and dataset can be retained.

In [8]:
# Schema required by the ASR pipeline.
pair_columns = ["pair_id", "original_path", "anonymized_path"]

# Example only:
# pairs_df = pd.DataFrame({
#     "pair_id": ["001", "002"],
#     "original_path": ["...", "..."],
#     "anonymized_path": ["...", "..."],
# })

print("Required columns:", pair_columns)


Required columns: ['pair_id', 'original_path', 'anonymized_path']


## 6. Transcribe both sides

For each pair:

$$
X_o\rightarrow W_o,
\qquad
X_a\rightarrow W_a.
$$

The paired transcripts are the main output of this notebook and feed Semantic Similarity, NLI, semantic alignment and TPI.

For the first real experiment, process a small subset (e.g. 100 pairs) before scaling up.

In [9]:
transcribe_pairs = asr.transcribe_pairs

## 7. Output for Content Evaluation

For each original/anonymized pair, we retain:

- original audio path;
- anonymized audio path;
- original transcript $W_o$;
- anonymized transcript $W_a$.

This notebook is responsible for **ASR and transcript generation only**. Semantic similarity, NLI, semantic alignment, and TPI are evaluated downstream.

### Next Step

Prepare the paired audio dataset with the required original and anonymized paths, then generate and save the corresponding transcripts. Use these outputs as inputs to the content-preservation evaluation pipeline.